imports & mounting drive

In [1]:
import numpy as np
import pandas as pd
import wandb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


data

In [2]:
data_path = '/content/drive/MyDrive/Ketastasia/data/dataset_seq15_ready.npz'
data = np.load(data_path)

X_train, y_train = data['X_train'], data['y_train']
X_val,   y_val    = data['X_val'],   data['y_val']
X_test,  y_test   = data['X_test'],  data['y_test']

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

import json
with open('/content/drive/MyDrive/Ketastasia/data/eda_results/eda_stats.json') as f:
    eda_stats = json.load(f)
class_names = eda_stats['classes']
print(f"{len(class_names)} classes: {class_names}")

# საბაზისო feature-სახელები (x_cols/y_cols/angle_cols იგივეა, რაც EDA notebook-ში)
x_cols = [f'x{i}' for i in range(13)]
y_cols = [f'y{i}' for i in range(13)]
angle_cols = ['angle_left_knee', 'angle_left_elbow', 'angle_left_hip']
base_cols = x_cols + y_cols + angle_cols

X_train: (2940, 15, 29), X_val: (580, 15, 29), X_test: (574, 15, 29)
25 classes: ['bench_press', 'bicep_curl', 'chest_fly', 'clean_and_jerk', 'deadlift', 'decline_bench_press', 'hammer_curl', 'hip_thrust', 'incline_bench_press', 'jump_rope', 'lat_pulldown', 'lateral_raise', 'leg_extension', 'leg_raises', 'plank', 'pullup', 'pushup', 'romanian_deadlift', 'russian_twist', 'shoulder_press', 'situp', 'squat', 't_bar_row', 'tricep_dips', 'tricep_pushdown']


label encoding

In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_idx = le.fit_transform(y_train)
y_val_idx   = le.transform(y_val)
y_test_idx  = le.transform(y_test)

print("Label encoding complete! First 5 samples:")
print("Text labels:   ", y_train[:5])
print("Integer indices:", y_train_idx[:5])

print("Unique encoded classes found in train:", np.unique(y_train_idx))

Label encoding complete! First 5 samples:
Text labels:    ['bench_press' 'bench_press' 'bench_press' 'bench_press' 'bench_press']
Integer indices: [0 0 0 0 0]
Unique encoded classes found in train: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24]


In [4]:
random_indices = np.random.choice(len(y_train), size=5, replace=False)

print("Random Samples Check:")
print("Text labels:   ", y_train[random_indices])
print("Integer indices:", y_train_idx[random_indices])

Random Samples Check:
Text labels:    ['bicep_curl' 'leg_raises' 'squat' 'incline_bench_press' 'squat']
Integer indices: [ 1 13 21  8 21]


feature engineering



In [5]:
def flatten_last_frame(X):
    """მხოლოდ სექვენციის ბოლო frame — მარტივი baseline"""
    return X[:, -1, :]

def flatten_aggregated(X):
    """mean/std/min/max თითო feature-ზე — ინახავს temporal dynamics-ს"""
    return np.concatenate([
        X.mean(axis=1), X.std(axis=1),
        X.min(axis=1),  X.max(axis=1)
    ], axis=1)

def flatten_full(X):
    """მთელი sequence ერთ გრძელ ვექტორად (15*29=435 feature)"""
    return X.reshape(X.shape[0], -1)

def get_feature_names(feature_name, base_cols, n_timesteps=15):
    """feature index -> წაკითხვადი სახელი, feature-ტიპის მიხედვით"""
    if feature_name == "aggregated":
        return ([f"mean_{c}" for c in base_cols] +
                [f"std_{c}" for c in base_cols] +
                [f"min_{c}" for c in base_cols] +
                [f"max_{c}" for c in base_cols])
    elif feature_name == "last_frame":
        return base_cols
    elif feature_name == "full_sequence":
        return [f"t{t}_{c}" for t in range(n_timesteps) for c in base_cols]
    else:
        raise ValueError(f"unknown feature_name: {feature_name}")

In [6]:
from google.colab import userdata
import wandb

wandb_api_key = userdata.get('WANDB_API_KEY_1')
wandb.login(key=wandb_api_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akeke23 (akeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

training

In [7]:
def run_rf_experiment(run_name, feature_fn, feature_name, rf_params, log_to_wandb=True):
    # Transform features based on strategy
    X_tr_feat  = feature_fn(X_train)
    X_val_feat = feature_fn(X_val)

    # Initialize and fit Random Forest
    rf = RandomForestClassifier(random_state=42, n_jobs=-1, **rf_params)
    rf.fit(X_tr_feat, y_train_idx)

    # Calculate train metrics
    train_pred = rf.predict(X_tr_feat)
    train_acc = accuracy_score(y_train_idx, train_pred)
    train_f1_macro = f1_score(y_train_idx, train_pred, average='macro')

    # Calculate validation metrics
    val_pred = rf.predict(X_val_feat)
    val_acc = accuracy_score(y_val_idx, val_pred)
    val_f1_macro = f1_score(y_val_idx, val_pred, average='macro')
    val_f1_weighted = f1_score(y_val_idx, val_pred, average='weighted')

    gap = train_f1_macro - val_f1_macro

    print(f"\n=== {run_name} ===")
    print(f"Features: {feature_name} (shape: {X_tr_feat.shape[1]})")
    print(f"Params: {rf_params}")
    print(f"Train Macro-F1: {train_f1_macro:.4f} | Val Macro-F1: {val_f1_macro:.4f} | Gap: {gap:.4f}")

    # Log results to WandB
    if log_to_wandb:
        wandb.init(
            project='ildolcefarniente',
            group='p1_randomforest',
            name=run_name,
            config={
                'model': 'random_forest',
                'feature_type': feature_name,
                'n_features': X_tr_feat.shape[1],
                **rf_params
            },
            reinit=True
        )
        wandb.run.summary['train_accuracy'] = train_acc
        wandb.run.summary['train_f1_macro'] = train_f1_macro
        wandb.run.summary['val_accuracy'] = val_acc
        wandb.run.summary['val_f1_macro'] = val_f1_macro
        wandb.run.summary['val_f1_weighted'] = val_f1_weighted
        wandb.run.summary['gap'] = gap

        # Feature Importance Logging
        feature_names = get_feature_names(feature_name, base_cols)
        importances = rf.feature_importances_
        top_idx = np.argsort(importances)[-15:][::-1]
        wandb.log({
            "feature_importance": wandb.plot.bar(
                wandb.Table(data=[[feature_names[i], importances[i]] for i in top_idx],
                            columns=["feature", "importance"]),
                "feature", "importance", title="Top 15 Feature Importances"
            )
        })

        # Confusion Matrix Logging (using alphabetical class names from JSON)
        wandb.log({
            "val_confusion_matrix": wandb.plot.confusion_matrix(
                probs=None, y_true=y_val_idx.tolist(), preds=val_pred.tolist(),
                class_names=class_names
            )
        })
        wandb.finish()

    return rf, val_acc, val_f1_macro, gap

In [8]:
experiments = [
    ("rf-lastframe-n100-dNone",      flatten_last_frame, "last_frame",   {'n_estimators': 100, 'max_depth': None}),
    ("rf-lastframe-n100-d3",         flatten_last_frame, "last_frame",   {'n_estimators': 100, 'max_depth': 3}),
    ("rf-agg-n100-dNone",            flatten_aggregated, "aggregated",   {'n_estimators': 100, 'max_depth': None}),
    ("rf-agg-n5-dNone",              flatten_aggregated, "aggregated",   {'n_estimators': 5, 'max_depth': None}),
    ("rf-agg-n20-d2",                flatten_aggregated, "aggregated",   {'n_estimators': 20, 'max_depth': 2}),
    ("rf-agg-n300-d20",              flatten_aggregated, "aggregated",   {'n_estimators': 300, 'max_depth': 20}),
    ("rf-agg-n300-dNone-minleaf1",   flatten_aggregated, "aggregated",   {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 1}),
    ("rf-agg-n300-dNone-minleaf10",  flatten_aggregated, "aggregated",   {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 10}),
    ("rf-agg-n500-dNone",            flatten_aggregated, "aggregated",   {'n_estimators': 500, 'max_depth': None}),
    ("rf-agg-n200-d15-balanced",     flatten_aggregated, "aggregated",   {'n_estimators': 200, 'max_depth': 15, 'class_weight': 'balanced'}),
    ("rf-fullseq-n200-d15",          flatten_full,        "full_sequence",{'n_estimators': 200, 'max_depth': 15}),
    ("rf-fullseq-n500-dNone-minleaf1", flatten_full,      "full_sequence",{'n_estimators': 500, 'max_depth': None, 'min_samples_leaf': 1}),
]

results = []
for run_name, feature_fn, feature_name, params in experiments:
    rf, val_acc, val_f1, gap = run_rf_experiment(run_name, feature_fn, feature_name, params)
    results.append({'run': run_name, 'features': feature_name, 'params': params,
                     'val_acc': val_acc, 'val_f1_macro': val_f1, 'gap': gap, 'model': rf})

results_df = pd.DataFrame([{k:v for k,v in r.items() if k != 'model'} for r in results])
results_df = results_df.sort_values('val_f1_macro', ascending=False)
print(results_df[['run', 'features', 'val_acc', 'val_f1_macro', 'gap']])


=== rf-lastframe-n100-dNone ===
Features: last_frame (shape: 29)
Params: {'n_estimators': 100, 'max_depth': None}
Train Macro-F1: 1.0000 | Val Macro-F1: 0.2956 | Gap: 0.7044


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


gap,0.70435
train_accuracy,1
train_f1_macro,1
val_accuracy,0.41724
val_f1_macro,0.29565
val_f1_weighted,0.4145



=== rf-lastframe-n100-d3 ===
Features: last_frame (shape: 29)
Params: {'n_estimators': 100, 'max_depth': 3}
Train Macro-F1: 0.0987 | Val Macro-F1: 0.0605 | Gap: 0.0382


gap,0.03825
train_accuracy,0.29116
train_f1_macro,0.09871
val_accuracy,0.23448
val_f1_macro,0.06046
val_f1_weighted,0.13436


...


=== rf-fullseq-n200-d15 ===
Features: full_sequence (shape: 435)
Params: {'n_estimators': 200, 'max_depth': 15}
Train Macro-F1: 0.9996 | Val Macro-F1: 0.4294 | Gap: 0.5702


gap,0.57018
train_accuracy,0.99932
train_f1_macro,0.99958
val_accuracy,0.57241
val_f1_macro,0.4294
val_f1_weighted,0.54187



=== rf-fullseq-n500-dNone-minleaf1 ===
Features: full_sequence (shape: 435)
Params: {'n_estimators': 500, 'max_depth': None, 'min_samples_leaf': 1}
Train Macro-F1: 1.0000 | Val Macro-F1: 0.3904 | Gap: 0.6096


gap,0.60965
train_accuracy,1
train_f1_macro,1
val_accuracy,0.5431
val_f1_macro,0.39035
val_f1_weighted,0.51183


                               run       features   val_acc  val_f1_macro  \
8                rf-agg-n500-dNone     aggregated  0.667241      0.587299   
6       rf-agg-n300-dNone-minleaf1     aggregated  0.663793      0.582962   
5                  rf-agg-n300-d20     aggregated  0.658621      0.577304   
7      rf-agg-n300-dNone-minleaf10     aggregated  0.646552      0.564134   
2                rf-agg-n100-dNone     aggregated  0.643103      0.559196   
9         rf-agg-n200-d15-balanced     aggregated  0.637931      0.548549   
10             rf-fullseq-n200-d15  full_sequence  0.572414      0.429404   
3                  rf-agg-n5-dNone     aggregated  0.520690      0.402514   
11  rf-fullseq-n500-dNone-minleaf1  full_sequence  0.543103      0.390350   
0          rf-lastframe-n100-dNone     last_frame  0.417241      0.295649   
4                    rf-agg-n20-d2     aggregated  0.253448      0.082390   
1             rf-lastframe-n100-d3     last_frame  0.234483      0.060463   

In [10]:
import os
import joblib
import wandb

# 1. Automatically find the best model from the results dataframe
best_run_row = results_df.iloc[0]
best_run_name = best_run_row['run']

# Extract the actual trained model object from the results list
best_rf_obj = next(item['model'] for item in results if item['run'] == best_run_name)

print(f"Best model in this notebook: {best_run_name}")
print(f"Validation F1-Macro: {best_run_row['val_f1_macro']:.4f}")

# 2. Save only the model object (without wrapper pipeline)
models_dir = '/content/drive/MyDrive/Ketastasia/models/'
os.makedirs(models_dir, exist_ok=True)
local_pkl_path = os.path.join(models_dir, 'pipeline1_best_rf.pkl')
joblib.dump(best_rf_obj, local_pkl_path)
print(f"Model saved locally at: {local_pkl_path}")

# 3. Register the model in the W&B Model Registry
run = wandb.init(project="ildolcefarniente", job_type="model_registration", name="register_pipeline1_rf")
artifact = wandb.Artifact(
    name="pipeline1_rf_models",
    type="model",
    description=f"RF models family. Best configuration: {best_run_name}"
)
artifact.add_file(local_pkl_path)
run.log_artifact(artifact, aliases=["latest"])
run.finish()

print("Model successfully uploaded to W&B registry.")

Best model in this notebook: rf-agg-n500-dNone
Validation F1-Macro: 0.5873
Model saved locally at: /content/drive/MyDrive/Ketastasia/models/pipeline1_best_rf.pkl


Model successfully uploaded to W&B registry.
